<div align="center"> 

---



# **Cerebellum Mega Analysis**


<img src="https://static.wixstatic.com/media/da4857_5f6dbdb6bbd84328b1e31c7f2dafb673~mv2.png/v1/fill/w_224,h_226,al_c,q_85,usm_0.66_1.00_0.01,enc_auto/enigma_addiction-300x249.png" alt="image" style="width:300px;"/>


 </img>
</div>


---
# <u> **Project Description:**</u>

The aim of this study is to perform a multi-site mega/meta analysis to assess the effects of substance use on volumetric changes in cerebellar regions.

## **Primary Analysis: Matchit Cross-Validated Permutation Tests**

This notebook performs the main analysis using matched samples (propensity score matching) with cross-validated permutation-based significance testing.

## **Sensitivity Analyses Included:**
- **CC (Complete-Cases)**: Subjects with complete Age/Sex data
- **AI (Alternative-Imputation)**: Age/Sex imputed by Substance + Site
- **CA (Covariate-Adjusted)**: ComBat harmonization with stratified covariate residualization

See `03-2_Permutation_Variants.ipynb` for comprehensive comparison of all three approaches.

## **Analysis Details**
   1. Main effects: MDTB cerebellar atlas
      - Cerebellar volume ~ substance group (User vs HC, Substance vs HC)
   2. Cross-validation: Group-stratified k-fold (by Database)
   3. Statistical testing: Permutation tests (1,000 permutations)
   4. Effect size: Cohen's d
   5. Multiple testing: FDR correction

---

In [1]:
addTaskCallback(function(...) {set.seed(42);TRUE})
options(warn=-1) # to turn off warnings
#options(warn=0) # to turn on warnings
options(repr.plot.width=10, repr.plot.height=6) # to resize plots

1 
1

In [2]:
setwd("/mnt/g/Other computers/My Laptop/PhD/Psilantro/Enigma_cerebellum/Enigma_Cerebellum/")

## Load Packages

In [3]:
# Load required packages  
  pacman::p_load(tidyverse,devtools,readxl,ggbeeswarm,gtExtras,gt,webshot,webshot2,VIM,
                ggpubr,cowplot,gtsummary,ff,ggthemes,scales,mice,vtreat,sjmisc,SmartEDA,boot,MKinfer,MatchIt,
                ggcorrplot,olsrr,lme4,rstatix,janitor,coin,rcartocolor,ggdist,gghalves,ggnewscale,patchwork,
                lsr,tidyr,reshape2,broom,irr,ggpubr,caret,tidymodels,foreach,doParallel,optmatch,emmeans)

# NeuroCombat
 if (!require("neuroCombat")){
   devtools::install_github("jfortin1/neuroCombat_Rpackage") }


# Load NeuroCombat
library(neuroCombat)

Loading required package: neuroCombat



## Set settings for plotting

In [4]:
theme_output  <- theme(text = element_text(family = "Times New Roman"),
                        axis.text.x = element_text(size = 20),
                        axis.text.y = element_text(size = 20),
                        axis.title.x = element_text(size = 25, face = "bold"),
                        axis.title.y = element_text(size = 25, face = "bold"),
                        legend.text = element_text(size = 22), legend.title = element_text(size = 24),
                        title = element_text(size = 20,face = "italic"))

pal_sex <- c(alpha("#8E5A9C",1),alpha("#2879BC",1))
pal_user <- c(alpha("#757575",1),alpha("#059DC7",1))
pal_substances <- c("HC" = alpha("#757575",1), "ALC" = alpha("#C93F4D",1), 
    "ATS" = alpha("#EB8F33",1), "CANN" = alpha("#0B8755",1), 
    "COC" = alpha("#3A5589",1), "NIC" = alpha("#764EA1",1),
    "OPI" = alpha("#06AED5",1))

In [5]:
`%notin%` <- Negate(`%in%`)

## Load data

In [6]:
load("Data/RData/Volumes_Cerebellum.RData")

# **1 Analysis for MDTB volumes**

In [7]:
num_perm <- 1000

## **1.1 Volume comparison of the full database**

### **1.1.1 Set data**

Final files:  
**MDTB_volumes**  
**MDTB_volumes_long** 

In [8]:
MDTB_volumes <- Volumes_Cerebellum_MDTB_Harmonized %>% 
                filter(Substance != "OPI") %>% 
                mutate(group = if_else(Substance == "HC",0,1) %>%
                        factor(labels = c("HC","User")),
                        Substance = factor(Substance, levels = c("ALC","ATS","CANN","COC","NIC","HC","OPI")), .before = 2) %>% 
                mutate(group = factor(group, levels = c("User","HC"))) %>%
                select(-c(Age,Sex,Control)) %>% unique() %>% 
                mutate(Database = factor(Database))

MDTB_cerebellar_regions_names <- MDTB_volumes %>%
    select(contains(c("region"))) %>% colnames()

# 1. transform the dataset to long format
MDTB_volumes_long <- MDTB_volumes %>% 
    pivot_longer(names_to = "region", values_to = "volume", MDTB_cerebellar_regions_names) %>%
    arrange(region) %>% mutate(region = factor(region, levels = unique(.$region)))

### **1.1.2 Users**

Final file: **p_values_MDTB_user_vs_control**

#### Running the linear model

In [9]:
## Compute the group effect with a linear model (HC as reference)
MDTB_lm_user_vs_control <- MDTB_volumes_long %>%
  select(group, region, volume) %>%
  nest(data = -region) %>%
  mutate(data = map(data, ~ .x %>% mutate(group = relevel(group, ref = "HC")))) %>%
  mutate(test = map(data, ~ {
    m <- lm(volume ~ group, data = .)
    dat <- broom::tidy(m, conf.int = TRUE) %>%
      dplyr::filter(term == "groupUser") %>%  # coefficient for User vs HC
      dplyr::mutate(
        df = df.residual(m),
        sigma = sigma(m))
    cont <- emmeans::emmeans(m, specs = "group") %>%
      emmeans::contrast(method = "pairwise")
    cont_df <- broom::tidy(cont)
    cohen_d <- cont_df$estimate / sigma(m)
    dat %>% mutate(effect.size = cohen_d)
  })) %>%
  unnest(test)


#### Obtain the null distributions

In [10]:
rm(MDTB_nulls_user_vs_control)
# Check in null distribution matrix exists 
if(exists("nulls_user_vs_control")) {
    print(1)
} else {
    MDTB_nulls_user_vs_control <- matrix(NA,ncol = MDTB_volumes %>% 
        select(MDTB_cerebellar_regions_names) %>% ncol(),nrow = num_perm)
}

In [11]:
MDTB_compute_null_distributions <- function(MDTB_nulls_user_vs_control) {
    if (sum(is.na(MDTB_nulls_user_vs_control)) > 0) {
        cl <- parallel::makeCluster(max(1, parallel::detectCores() - 1))
        on.exit(parallel::stopCluster(cl), add = TRUE)
        doParallel::registerDoParallel(cl)

        res <- foreach(i = seq_len(num_perm),
                       .packages = c("dplyr", "tidyr", "purrr", "broom"),
                       .export = c("MDTB_volumes", "MDTB_cerebellar_regions_names", "num_perm")) %dopar% {
            set.seed(42 + i)
            boot <- sample(seq_len(nrow(MDTB_volumes)), size = nrow(MDTB_volumes), replace = TRUE)

            x <- tibble(
                    group = stats::relevel(MDTB_volumes$group[boot], ref = "HC")
                ) %>%
                dplyr::bind_cols(dplyr::select(MDTB_volumes, dplyr::all_of(MDTB_cerebellar_regions_names))) %>%
                tidyr::pivot_longer(cols = dplyr::all_of(MDTB_cerebellar_regions_names),
                                    names_to = "region",
                                    values_to = "volume") %>%
                tidyr::nest(data = -region) %>%
                dplyr::mutate(test = purrr::map(data, ~ {
                    m <- lm(volume ~ group, data = .x)
                    broom::tidy(m) %>%
                        dplyr::filter(term == "groupUser") %>%
                        dplyr::select(statistic)
                })) %>%
                tidyr::unnest(test) %>%
                dplyr::pull(statistic)

            x
        }

        for (j in seq_len(num_perm)) {
            MDTB_nulls_user_vs_control[j, ] <- res[[j]]
        }
    }
    return(MDTB_nulls_user_vs_control)
}

#### Perform it

In [12]:
dir.create("Statistics/Linear_models/MDTB/RData", showWarnings = FALSE, recursive = TRUE)

nulls_fp <- file.path("Statistics", "Linear_models", "MDTB", "RData", "MDTB_nulls_user_vs_control.RData")

if (!file.exists(nulls_fp)) {

    MDTB_nulls_user_vs_control <- MDTB_compute_null_distributions(MDTB_nulls_user_vs_control)

    # add names to the nulls and transform to a tibble
    colnames(MDTB_nulls_user_vs_control) <- MDTB_lm_user_vs_control$region

    ci_lower <- apply(MDTB_nulls_user_vs_control, 2, quantile, probs = 0.025)  
    ci_upper <- apply(MDTB_nulls_user_vs_control, 2, quantile, probs = 0.975)

    MDTB_lm_user_vs_control$perm_CI_lower <- ci_lower 
    MDTB_lm_user_vs_control$perm_CI_upper <- ci_upper

    # # Calculate the p-values

    # Calculate permutation p-values aligned by region
    pvals <- purrr::map2_dbl(
        as.data.frame(MDTB_nulls_user_vs_control),
        MDTB_lm_user_vs_control$statistic, 
        ~ mean(abs(.x) >= abs(.y))
    )

    MDTB_lm_user_vs_control_permuted <- MDTB_lm_user_vs_control %>%
        dplyr::select(-data) %>%
        dplyr::mutate(
            p.value = pvals,
            perm_CI_lower = ci_lower[as.character(region)],
            perm_CI_upper = ci_upper[as.character(region)]
        )

    p_values_MDTB_user_vs_control <- MDTB_lm_user_vs_control_permuted %>%
        dplyr::mutate(p.adj = p.adjust(p.value, method = "fdr"))

    save(MDTB_nulls_user_vs_control,MDTB_lm_user_vs_control_permuted,p_values_MDTB_user_vs_control, file = nulls_fp)
} else {
    load(nulls_fp)
}

### **1.1.3 Substances**

Final file: **p_values_MDTB_substance_vs_control**

#### Running the linear models

In [13]:
# Define a vector with the names of the substance
substances_names  <- unique(MDTB_volumes$Substance)[unique(MDTB_volumes$Substance) !="HC"] %>% as.vector()

In [14]:
# Now that we've compared the HC groups lets do it for each HC vs each substance
## Compute the group effect with a linear model (HC as reference)

MDTB_lm_substance_vs_control <- map_df(substances_names, .f = ~ {
    MDTB_volumes_long %>%
        dplyr::filter(Substance %in% c("HC", .x)) %>%
        dplyr::select(group, region, volume) %>%
        tidyr::nest(data = -region) %>%
        dplyr::mutate(data = purrr::map(data, ~ .x %>% dplyr::mutate(group = relevel(group, ref = "HC")))) %>%
        dplyr::mutate(test = purrr::map(data, ~ {
            m <- lm(volume ~ group, data = .)
            dat <- broom::tidy(m, conf.int = TRUE) %>%
                dplyr::filter(term == "groupUser") %>%
                dplyr::mutate(
                    df = df.residual(m),
                    sigma = sigma(m)
                )
            cont <- emmeans::emmeans(m, specs = "group") %>%
                emmeans::contrast(method = "pairwise")
            cont_df <- broom::tidy(cont)
            cohen_d <- cont_df$estimate / sigma(m)
            dat %>% dplyr::mutate(effect.size = cohen_d)
        })) %>%
        tidyr::unnest(test) %>%
        dplyr::mutate(Substance = .x)
})

#### Obtain the null distributions

In [15]:
# calculate the p.values with bootstrapped CI
# Create a list to store all the null distributions
if (!exists("MDTB_nulls_substance_vs_control")){

MDTB_nulls_substance_vs_control <- lapply(1:length(substances_names),
    matrix, data = NA,
    nrow = num_perm, 
    ncol = MDTB_volumes %>% select(MDTB_cerebellar_regions_names) %>% ncol()
    ) %>%
setNames(substances_names)
}


In [16]:
# Parallel permutation function (foreach) for substances vs control
MDTB_compute_null_distributions_substances <- function(MDTB_volumes,
                              MDTB_cerebellar_regions_names,
                              substances_names,
                              MDTB_nulls_substance_vs_control,
                              num_perm) {
  if (sum(is.na(unlist(MDTB_nulls_substance_vs_control))) == 0) return(MDTB_nulls_substance_vs_control)

  cl <- parallel::makeCluster(max(1, parallel::detectCores() - 1))
  on.exit(parallel::stopCluster(cl), add = TRUE)
  doParallel::registerDoParallel(cl)

  res <- foreach(i = seq_along(substances_names),
         .packages = c("dplyr", "tidyr", "purrr", "broom"),
         .export = c("MDTB_volumes", "MDTB_cerebellar_regions_names", "substances_names", "MDTB_nulls_substance_vs_control", "num_perm")) %dopar% {
  # If this substance already has values, return as is
  current <- MDTB_nulls_substance_vs_control[[i]]
  if (!is.null(current) && sum(is.na(current)) == 0) return(current)

  set.seed(42 + i)
  n_cols <- MDTB_volumes %>% dplyr::select(dplyr::all_of(MDTB_cerebellar_regions_names)) %>% ncol()
  nulls <- matrix(NA_real_, nrow = num_perm, ncol = n_cols)

  sub_df <- MDTB_volumes %>%
    dplyr::filter(Substance %in% c("HC", substances_names[i])) %>%
    dplyr::select(Substance, group, dplyr::all_of(MDTB_cerebellar_regions_names))

  for (j in seq_len(num_perm)) {
    boot <- sample.int(n = nrow(sub_df), replace = TRUE)

    y <- dplyr::bind_cols(
      data.frame(
      group = sub_df$group[boot],
      check.names = FALSE
      ),
      dplyr::select(sub_df, dplyr::all_of(MDTB_cerebellar_regions_names))
       ) %>%
    dplyr::mutate(group = stats::relevel(group, ref = "HC")) %>%
    tidyr::pivot_longer(cols = dplyr::all_of(MDTB_cerebellar_regions_names),
        names_to = "region", values_to = "volume") %>%
    tidyr::nest(data = -region) %>%
    dplyr::mutate(test = purrr::map(data, ~ {
      m <- lm(volume ~ group, data = .x)
      broom::tidy(m) %>%
      dplyr::filter(term == "groupUser") %>%
      dplyr::select(statistic)
    })) %>%
    tidyr::unnest(test) %>%
    dplyr::pull(statistic)

    nulls[j, ] <- y
  }
  nulls
  }

  res <- res %>% rlang::set_names(substances_names)
  return(res)
}

#### Perform it

In [17]:
dir.create("Statistics/Linear_models/MDTB/RData", showWarnings = FALSE, recursive = TRUE)

# Save/load null distributions for substances vs control
nulls_fp_substances <- file.path("Statistics", "Linear_models", "MDTB", "RData",
                 "MDTB_nulls_substance_vs_control.RData")

if (!file.exists(nulls_fp_substances)) {
  MDTB_nulls_substance_vs_control <- MDTB_compute_null_distributions_substances(
  MDTB_volumes,
  MDTB_cerebellar_regions_names,
  substances_names,
  MDTB_nulls_substance_vs_control,
  num_perm)

  # Ensure null matrices have region column names
  MDTB_nulls_substance_vs_control <- purrr::imap(
    MDTB_nulls_substance_vs_control,
    ~ { colnames(.x) <- MDTB_cerebellar_regions_names; .x }
  )

  # Rebuild CIs per Substance x region from null distributions
  MDTB_CIs <- purrr::imap_dfr(
    MDTB_nulls_substance_vs_control,
    ~ {
      low  <- apply(.x, 2, quantile, probs = 0.025)
      high <- apply(.x, 2, quantile, probs = 0.975)
      tibble(
        Substance = .y,
        region    = names(low),
        perm_CI_lower    = as.numeric(low),
        perm_CI_upper     = as.numeric(high)
      )})

  # Compute permutation p-values from nulls and merge with CIs
  p_values_MDTB_substance_vs_control <- MDTB_lm_substance_vs_control %>%
    dplyr::select(-data) %>%
    dplyr::mutate(
      p_value = purrr::pmap_dbl(
        list(Substance, as.character(region), statistic),
        function(substance, roi, stat) {
          null_distribution <- MDTB_nulls_substance_vs_control[[substance]][, roi]
          mean(abs(null_distribution) >= abs(stat))
        })
    ) %>%
    dplyr::group_by(Substance) %>%
    dplyr::mutate(p_adj = p.adjust(p_value, method = "fdr")) %>%
    dplyr::ungroup() %>%
    dplyr::left_join(MDTB_CIs, by = c("Substance", "region"))
    
  save(MDTB_nulls_substance_vs_control,MDTB_CIs,p_values_MDTB_substance_vs_control, file = nulls_fp_substances)
} else {
  load(nulls_fp_substances)
}

## **1.2 Volume comparison by matching sample size**

### **1.2.1 Users**

#### **1.2.1.1 Set data and matchid**

Final file: **MDTB_volumes_matchit_users_df**

In [18]:
MDTB_cerebellar_regions_names <- MDTB_volumes %>%
    select(contains(c("region"))) %>% colnames()

MDTB_volumes_matchit_users_raw <- Volumes_Cerebellum_MDTB_raw %>% 
    mutate(Substance = case_when(Control != 0 ~ Substance, Control == 0 ~ "HC")) %>% 
    filter(!is.na(Substance)) %>% 
    mutate(Sex = factor(Sex,levels = c("0","1")),
        group = if_else(Substance == "HC",0,1) %>%
                        factor(labels = c("HC","User")),
        group = factor(group, levels = c("User","HC")),
        Substance = factor(Substance, levels = c("ALC","ATS","CANN","COC","NIC","OPI","HC")), .before = 2) %>%
    select(-c(Control))

# Match sample for each substance
MDTB_volumes_matchit_model_users_raw <- MDTB_volumes_matchit_users_raw %>% 
        matchit(formula = group ~ Age + Sex, 
                method = "nearest", distance = "glm", link = "probit",
                ratio = 1, replace = F)

MDTB_volumes_matchit_users_df <- match.data(MDTB_volumes_matchit_model_users_raw) %>% 
        select(-c(distance,weights,subclass)) %>% 
        mutate(Database = factor(Database))

#### **1.2.1.2 Harmonize and prepare data**

Final file: **MDTB_volumes_matchit_users_harmonized_df**  
**MDTB_volumes_matchit_users_harmonized_long**

In [19]:
# Harmonization volumes controling for Age, Sex
# Note: volumes are already corrected by eTIV

MDTB_volumes_matchit_users_harmonized_df <- MDTB_volumes_matchit_users_df %>% list() %>% 
  map(~ {
    df  <- .x %>% mutate(Database = factor(Database))
    roi <- intersect(MDTB_cerebellar_regions_names, colnames(df))
    mod <- model.matrix(~ Age + Sex + eTIV, data = df)

    combat <- neuroCombat(
      dat        = t(as.matrix(dplyr::select(df, dplyr::all_of(roi)))),
      batch      = df$Database,
      mod        = mod,
      mean.only  = TRUE,
      parametric = TRUE
    )

    vol_harm <- t(combat$dat.combat) %>% as_tibble()
    colnames(vol_harm) <- roi

    dplyr::bind_cols(dplyr::select(df, -dplyr::all_of(roi)), vol_harm) %>% select(-c(eTIV))
  }) %>% reduce(tibble)

[neuroCombat] Found 640 missing data values. 
[neuroCombat] Performing ComBat with mean only
[neuroCombat] Performing ComBat with empirical Bayes
[neuroCombat] Found 55 batches
[neuroCombat] Adjusting for  3  covariate(s) or covariate level(s)
[neuroCombat] Standardizing Data across features
[neuroCombat] Fitting L/S model and finding priors
[neuroCombat] Finding parametric adjustments
[neuroCombat] Adjusting the Data


In [20]:
# transform the dataset to long format
MDTB_volumes_matchit_users_harmonized_long <- MDTB_volumes_matchit_users_harmonized_df %>% 
    pivot_longer(names_to = "region", values_to = "volume", MDTB_cerebellar_regions_names) %>%
    arrange(region) %>% mutate(region = factor(region, levels = unique(.$region)))

#### **1.2.1.3 Linear models**

Final file: **p_values_MDTB_matchit_users_vs_control**

##### Running the linear models

In [21]:
# Now that we've compared the HC groups lets do it for each HC vs each substance
## Compute the group effect with a linear model (HC as reference)

MDTB_lm_matchit_users_vs_control <- MDTB_volumes_matchit_users_harmonized_long %>%
    select(group, region, volume) %>%
    nest(data = -region) %>%
    mutate(data = map(data, ~ .x %>% mutate(group = relevel(group, ref = "HC")))) %>%
    mutate(test = map(data, ~ {
        m <- lm(volume ~ group, data = .)
        dat <- broom::tidy(m, conf.int = TRUE) %>%
            dplyr::filter(term == "groupUser") %>%
            dplyr::mutate(
                df = df.residual(m),
                sigma = sigma(m)
            )
        cont <- emmeans::emmeans(m, specs = "group") %>%
            emmeans::contrast(method = "pairwise")
        cont_df <- broom::tidy(cont)
        cohen_d <- cont_df$estimate / sigma(m)
        dat %>% mutate(effect.size = cohen_d)
    })) %>%
    unnest(test)

##### Obtain the null distributions

In [22]:
# calculate the p.values with bootstrapped CI
# Create a list to store all the null distributions

if (!exists("MDTB_nulls_matchit_users_vs_control")) {
MDTB_nulls_matchit_users_vs_control <- matrix(NA,ncol = MDTB_volumes_matchit_users_harmonized_df %>% 
    select(MDTB_cerebellar_regions_names) %>% ncol(),nrow = num_perm)
}

In [23]:
# Parallel permutation function (foreach) for users vs control

MDTB_compute_null_distributions_matchit_users <- function(MDTB_volumes_matchit_users_harmonized_df,
                              MDTB_cerebellar_regions_names,
                              MDTB_nulls_matchit_users_vs_control) {
  if (sum(is.na(MDTB_nulls_matchit_users_vs_control)) > 0) {
  cl <- parallel::makeCluster(max(1, parallel::detectCores() - 1))
  on.exit(parallel::stopCluster(cl), add = TRUE)
  doParallel::registerDoParallel(cl)

  res <- foreach(i = seq_len(num_perm),
           .packages = c("dplyr", "tidyr", "purrr", "broom"),
           .export = c("MDTB_volumes_matchit_users_harmonized_df", "MDTB_cerebellar_regions_names", "num_perm")) %dopar% {
    set.seed(42 + i)
    boot <- sample.int(n = nrow(MDTB_volumes_matchit_users_harmonized_df), replace = TRUE)

    x <- dplyr::bind_cols(
    data.frame(
      group = MDTB_volumes_matchit_users_harmonized_df$group[boot],
      check.names = FALSE
    ),
    dplyr::select(MDTB_volumes_matchit_users_harmonized_df, dplyr::all_of(MDTB_cerebellar_regions_names))
    ) %>%
    dplyr::mutate(group = stats::relevel(as.factor(group), ref = "HC")) %>%  # set HC as reference
    tidyr::pivot_longer(cols = dplyr::all_of(MDTB_cerebellar_regions_names),
              names_to = "region", values_to = "volume") %>%
    tidyr::nest(data = -region) %>%
    dplyr::mutate(test = purrr::map(data, ~ {
      m <- lm(volume ~ group, data = .x)
      broom::tidy(m) %>%
      dplyr::filter(term == "groupUser") %>%  # coefficient for User vs HC
      dplyr::select(statistic)
    })) %>%
    tidyr::unnest(test) %>%
    dplyr::pull(statistic)

    x
  }

  for (j in seq_len(num_perm)) {
    MDTB_nulls_matchit_users_vs_control[j, ] <- res[[j]]
  }
  }
  return(MDTB_nulls_matchit_users_vs_control)
}


##### Perform it

In [24]:

dir.create("Statistics/Linear_models/MDTB/RData", showWarnings = FALSE, recursive = TRUE)
# Save/load null distributions for users vs control

nulls_fp_matchit_users <- file.path("Statistics", "Linear_models", "MDTB", "RData",
                 "MDTB_nulls_matchit_users_vs_control.RData")

if (!file.exists(nulls_fp_matchit_users)) {
    MDTB_nulls_matchit_users_vs_control <- MDTB_compute_null_distributions_matchit_users(
        MDTB_volumes_matchit_users_harmonized_df,
        MDTB_cerebellar_regions_names,
        MDTB_nulls_matchit_users_vs_control)
    # add names to the nulls and transform to a tibble
    colnames(MDTB_nulls_matchit_users_vs_control) <- as.character(MDTB_lm_matchit_users_vs_control$region)
    
    ci_lower <- apply(MDTB_nulls_matchit_users_vs_control, 2, quantile, probs = 0.025)  
    ci_upper <- apply(MDTB_nulls_matchit_users_vs_control, 2, quantile, probs = 0.975)
    
    # Calculate permutation p-values aligned by region
    pvals <- purrr::map2_dbl(
        as.data.frame(MDTB_nulls_matchit_users_vs_control),
        MDTB_lm_matchit_users_vs_control$statistic, 
        ~ mean(abs(.x) >= abs(.y))
    )
    
    MDTB_lm_matchit_users_vs_control_permuted <- MDTB_lm_matchit_users_vs_control %>%
        select(-data) %>%
        mutate(
            p.value = pvals,
            perm_CI_lower = ci_lower[as.character(region)],
            perm_CI_upper = ci_upper[as.character(region)]
        )
    
    p_values_MDTB_matchit_users_vs_control <- MDTB_lm_matchit_users_vs_control_permuted %>%
        mutate(p.adj = p.adjust(p.value, method = "fdr"))
    
    save(MDTB_nulls_matchit_users_vs_control,
        MDTB_lm_matchit_users_vs_control_permuted,
        p_values_MDTB_matchit_users_vs_control, 
        file = nulls_fp_matchit_users)
    } else {
    load(nulls_fp_matchit_users)
    }

### **1.2.2 Substances**

#### **1.2.2.1 Set data and matchid**

Final file: **MDTB_volumes_matchit_substances_df**

In [25]:
# Build raw dataset (do not redefine substances_names or MDTB_cerebellar_regions_names)
MDTB_volumes_matchit_substances_raw <- Volumes_Cerebellum_MDTB_raw %>% 
    mutate(Substance = if_else(Control == 0, "HC", as.character(Substance))) %>% 
    filter(!is.na(Substance)) %>% 
    mutate(
        Sex = factor(Sex, levels = c("0","1")),
        group = if_else(Substance == "HC", 0, 1) %>% factor(labels = c("HC","User")),
        group = factor(group, levels = c("User","HC")),
        Substance = factor(Substance, levels = c("ALC","ATS","CANN","COC","NIC","OPI","HC")),
        Database = factor(Database)
    ) %>%
    select(-Control)

set.seed(42)
# Prioritize HC from same Database; supplement with HC from other Databases if needed
MDTB_volumes_matchit_model_substances_raw <- substances_names %>%
    purrr::map(~ {
        cases <- MDTB_volumes_matchit_substances_raw %>% dplyr::filter(Substance == .x)
        dbs_cases <- unique(cases$Database)
        hc_same <- MDTB_volumes_matchit_substances_raw %>%
            dplyr::filter(Substance == "HC", Database %in% dbs_cases)

        needed <- nrow(cases) - nrow(hc_same)

        if (needed > 0) {
            other_hc <- MDTB_volumes_matchit_substances_raw %>%
                dplyr::filter(Substance == "HC", !Database %in% dbs_cases)

            db_counts <- other_hc %>%
                dplyr::count(Database, name = "n") %>%
                dplyr::arrange(dplyr::desc(n))

            selected_hc <- tibble()
            remaining <- needed

            for (i in seq_len(nrow(db_counts))) {
                db_i <- db_counts$Database[i]
                db_controls <- other_hc %>% dplyr::filter(Database == db_i)
                if (remaining <= nrow(db_controls)) {
                    selected_hc <- dplyr::bind_rows(
                        selected_hc,
                        db_controls %>% dplyr::slice_sample(n = remaining)
                    )
                    remaining <- 0
                    break
                } else {
                    selected_hc <- dplyr::bind_rows(selected_hc, db_controls)
                    remaining <- needed - nrow(selected_hc)
                }
            }

            if (remaining > 0) {
                # Not enough controls; keep all we have
                controls_pool <- dplyr::bind_rows(hc_same, selected_hc)
            } else {
                controls_pool <- dplyr::bind_rows(hc_same, selected_hc)
            }
        } else {
            controls_pool <- hc_same %>% dplyr::slice_sample(n = nrow(cases))
        }

        analysis_df <- dplyr::bind_rows(cases, controls_pool)

        matchit(
            formula = group ~ Age + Sex,
            data = analysis_df,
            method = "nearest",
            distance = "glm",
            link = "probit",
            ratio = 1,
            replace = FALSE
        )
    }) %>%
    rlang::set_names(substances_names)

MDTB_volumes_matchit_substances_df <- MDTB_volumes_matchit_model_substances_raw %>%
    purrr::map(~ match.data(.x) %>%
        dplyr::select(-distance, -weights, -subclass) %>%
        dplyr::mutate(Database = factor(Database)))

#### **1.2.2.2 Harmonize and prepare data**

Final file:  
**MDTB_volumes_matchit_substances_harmonized_df**   
**MDTB_volumes_matchit_substances_harmonized_long**

In [26]:
# Harmonization volumes controling for Age, Sex

MDTB_volumes_matchit_substances_harmonized_df <- MDTB_volumes_matchit_substances_df %>% 
  map(~ {
    df  <- .x %>% mutate(Database = factor(Database))
    roi <- intersect(MDTB_cerebellar_regions_names, colnames(df))
    mod <- model.matrix(~ Age + Sex + eTIV, data = df)

    combat <- neuroCombat(
      dat        = t(as.matrix(dplyr::select(df, dplyr::all_of(roi)))),
      batch      = df$Database,
      mod        = mod,
      mean.only  = TRUE,
      parametric = TRUE
    )

    vol_harm <- t(combat$dat.combat) %>% as_tibble()
    colnames(vol_harm) <- roi

    dplyr::bind_cols(dplyr::select(df, -dplyr::all_of(roi)), vol_harm)
  })

[neuroCombat] Found 420 missing data values. 
[neuroCombat] Performing ComBat with mean only
[neuroCombat] Performing ComBat with empirical Bayes
[neuroCombat] Found 25 batches
[neuroCombat] Adjusting for  3  covariate(s) or covariate level(s)
[neuroCombat] Standardizing Data across features
[neuroCombat] Fitting L/S model and finding priors
[neuroCombat] Finding parametric adjustments
[neuroCombat] Adjusting the Data
[neuroCombat] Performing ComBat with mean only
[neuroCombat] Performing ComBat with empirical Bayes
[neuroCombat] Found 6 batches
[neuroCombat] Adjusting for  3  covariate(s) or covariate level(s)
[neuroCombat] Standardizing Data across features
[neuroCombat] Fitting L/S model and finding priors
[neuroCombat] Finding parametric adjustments
[neuroCombat] Adjusting the Data
[neuroCombat] Found 770 missing data values. 
[neuroCombat] Performing ComBat with mean only
[neuroCombat] Performing ComBat with empirical Bayes
[neuroCombat] Found 11 batches
[neuroCombat] Adjusting fo

In [27]:
# transform the dataset to long format
MDTB_volumes_matchit_substances_harmonized_long <- MDTB_volumes_matchit_substances_harmonized_df %>% 
    map(~ .x %>%
    pivot_longer(names_to = "region", values_to = "volume", MDTB_cerebellar_regions_names) %>%
    arrange(region) %>% mutate(region = factor(region, levels = unique(.$region))) %>%
    select(-c(Sex,Age)) )

#### **1.2.2.3 Linear models**

Final file: **p_values_MDTB_matchit_substance_vs_control**

##### Running the linear models

In [28]:
# Now that we've compared the HC groups lets do it for each HC vs each substance
## Compute the group effect with HC as reference

MDTB_lm_matchit_substance_vs_control <- map_df(
    substances_names,
    .f = ~ MDTB_volumes_matchit_substances_harmonized_long[[.x]] %>%
        dplyr::filter(Substance %in% c("HC", .x)) %>%
        dplyr::select(group, region, volume) %>%
        tidyr::nest(data = -region) %>%
        dplyr::mutate(data = purrr::map(data, ~ dplyr::mutate(.x, group = stats::relevel(group, ref = "HC")))) %>%
        dplyr::mutate(test = purrr::map(data, ~ {
            m <- lm(volume ~ group, data = .)
            dat <- broom::tidy(m, conf.int = TRUE) %>%
                dplyr::filter(term == "groupUser") %>%
                dplyr::mutate(
                    df = df.residual(m),
                    sigma = sigma(m)
                )
            cont <- emmeans::emmeans(m, specs = "group") %>%
                emmeans::contrast(method = "pairwise")
            cont_df <- broom::tidy(cont)
            cohen_d <- cont_df$estimate / sigma(m)
            dat %>% dplyr::mutate(effect.size = cohen_d)
        })) %>%
        tidyr::unnest(test) %>%
        dplyr::mutate(Substance = .x)
)

##### Obtain the null distributions

In [29]:
# calculate the p.values with bootstrapped CI
# Create a list to store all the null distributions
if (!exists("MDTB_nulls_matchit_substance_vs_control")){
    MDTB_nulls_matchit_substance_vs_control <- lapply(1:length(substances_names),
        matrix, data = NA,
        nrow = num_perm, 
        ncol = MDTB_volumes %>% select(MDTB_cerebellar_regions_names) %>% ncol()
        ) %>%
    setNames(substances_names)
}

In [30]:
# Parallel permutation function (foreach) for substances vs control
MDTB_compute_null_distributions_matchit_substances <- function(MDTB_volumes,
                              MDTB_cerebellar_regions_names,
                              substances_names,
                              MDTB_nulls_matchit_substance_vs_control,
                              num_perm) {
  if (sum(is.na(unlist(MDTB_nulls_matchit_substance_vs_control))) == 0) return(MDTB_nulls_matchit_substance_vs_control)

  cl <- parallel::makeCluster(max(1, parallel::detectCores() - 1))
  on.exit(parallel::stopCluster(cl), add = TRUE)
  doParallel::registerDoParallel(cl)

  res <- foreach(i = seq_along(substances_names),
         .packages = c("dplyr", "tidyr", "purrr", "broom"),
         .export = c("MDTB_volumes", "MDTB_cerebellar_regions_names", "substances_names", "MDTB_nulls_matchit_substance_vs_control", "num_perm")) %dopar% {
  # If this substance already has values, return as is
  current <- MDTB_nulls_matchit_substance_vs_control[[i]]
  if (!is.null(current) && sum(is.na(current)) == 0) return(current)

  set.seed(42 + i)
  n_cols <- MDTB_volumes %>% dplyr::select(dplyr::all_of(MDTB_cerebellar_regions_names)) %>% ncol()
  nulls <- matrix(NA_real_, nrow = num_perm, ncol = n_cols)

  sub_df <- MDTB_volumes %>%
    dplyr::filter(Substance %in% c("HC", substances_names[i])) %>%
    dplyr::select(Substance, group, dplyr::all_of(MDTB_cerebellar_regions_names))

  for (j in seq_len(num_perm)) {
    boot <- sample.int(n = nrow(sub_df), replace = TRUE)

    y <- dplyr::bind_cols(
      data.frame(
      group = sub_df$group[boot],
      check.names = FALSE
      ),
      dplyr::select(sub_df, dplyr::all_of(MDTB_cerebellar_regions_names))
    ) %>%
    dplyr::mutate(group = stats::relevel(as.factor(group), ref = "HC")) %>%  # set HC as reference
    tidyr::pivot_longer(cols = dplyr::all_of(MDTB_cerebellar_regions_names),
        names_to = "region", values_to = "volume") %>%
    tidyr::nest(data = -region) %>%
    dplyr::mutate(test = purrr::map(data, ~ {
      m <- lm(volume ~ group, data = .x)
      broom::tidy(m) %>%
      dplyr::filter(term == "groupUser") %>%  # coefficient for User vs HC
      dplyr::select(statistic)
    })) %>%
    tidyr::unnest(test) %>%
    dplyr::pull(statistic)
    nulls[j, ] <- y
  }
  nulls
  }

  res <- res %>% rlang::set_names(substances_names)
  return(res)
}

##### Perform it

In [31]:

dir.create("Statistics/Linear_models/MDTB/RData", showWarnings = FALSE, recursive = TRUE)
# Save/load null distributions for substances vs control
nulls_fp_matchit_substances <- file.path("Statistics", "Linear_models", "MDTB", "RData",
                 "MDTB_nulls_matchit_substance_vs_control.RData")
if (!file.exists(nulls_fp_matchit_substances)) {
    MDTB_nulls_matchit_substance_vs_control <- MDTB_compute_null_distributions_matchit_substances(
    MDTB_volumes,
    MDTB_cerebellar_regions_names,
    substances_names,
    MDTB_nulls_matchit_substance_vs_control,
    num_perm)
    # Ensure null matrices have region column names
    MDTB_nulls_matchit_substance_vs_control <- purrr::imap(
      MDTB_nulls_matchit_substance_vs_control,
      ~ { colnames(.x) <- MDTB_cerebellar_regions_names; .x }
    )
    # Rebuild CIs per Substance x region from null distributions
    MDTB_matchit_CIs <- purrr::imap_dfr(
      MDTB_nulls_matchit_substance_vs_control,
      ~ {
        low  <- apply(.x, 2, quantile, probs = 0.025)
        high <- apply(.x, 2, quantile, probs = 0.975)
        tibble(
          Substance = .y,
          region    = names(low),
          perm_CI_lower    = as.numeric(low),
          perm_CI_upper     = as.numeric(high)
        )})
    # Compute permutation p-values from nulls and merge with CIs
    p_values_MDTB_matchit_substance_vs_control <- MDTB_lm_matchit_substance_vs_control %>%
      dplyr::select(-data) %>%
      dplyr::mutate(
        p_value = purrr::pmap_dbl(
          list(Substance, as.character(region), statistic),
          function(substance, roi, stat) {
            null_distribution <- MDTB_nulls_matchit_substance_vs_control[[substance]][, roi]
            mean(abs(null_distribution) >= abs(stat))
          })
      ) %>%
      dplyr::group_by(Substance) %>%
      dplyr::mutate(p_adj = p.adjust(p_value, method = "fdr")) %>%
      dplyr::ungroup() %>%
      dplyr::left_join(MDTB_matchit_CIs, by = c("Substance", "region"))

    save(MDTB_nulls_matchit_substance_vs_control,MDTB_matchit_CIs,
        p_values_MDTB_matchit_substance_vs_control, file = nulls_fp_matchit_substances)
} else {
    load(nulls_fp_matchit_substances)
}

## **1.3 Volume comparison with permutation by k-fold cross-validation**

### **1.3.1 Exploring the databases**

plots: 
**plot_subs_per_DB**

In [32]:
# Count subjects per Database x Substance (including HC)
subjects_per_db_substance <- Volumes_Cerebellum_MDTB_raw %>%
    mutate(Substance = if_else(Control == 0, "HC", as.character(Substance))) %>%
    filter(!is.na(Substance)) %>%
    mutate(Substance = factor(Substance, levels = c("ALC","ATS","CANN","COC","NIC","OPI","HC"))) %>%
    group_by(Database, Substance) %>%
    summarise(n_subjects = dplyr::n_distinct(ID_MRI), .groups = "drop") %>%
    arrange(Database, Substance)

plot_subs_per_DB <- subjects_per_db_substance %>%
    ggplot(aes(x = n_subjects,
                         y = forcats::fct_reorder(Database, n_subjects, .fun = sum, .desc = TRUE),
                         fill = Substance)) +
    geom_col(width = 0.7) +
    scale_fill_manual(values = pal_substances, drop = FALSE) +
    scale_x_continuous(expand = c(0, 0)) +
    labs(title = "Subjects per Database by Substance",
             x = "Number of subjects",
             y = "Database",
             fill = "Substance") +
    theme_output +
    theme(
      panel.grid.major.y = element_blank(),
      axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, size = 20)
    ) + coord_flip()

ggsave(plot = plot_subs_per_DB,filename = "Figures/Supplementaries/Figure_S1/Subjects_per_Database_by_Substance_MDTB.png",
        width = 50, height=20, units="cm", bg = "white", dpi = 300)

### **1.3.2 splitting the k-folds**

Final file:  
**MDTB_volumes_raw_folds_list**  

In [33]:
MDTB_volumes_raw_folds_set <- Volumes_Cerebellum_MDTB_raw

# Leave-one-database-out per fold 
# Determine number of folds as number of unique databases
kfolds <- nlevels(MDTB_volumes_raw_folds_set$Database)

# Leave-one-database-out cross-validation using rsample
MDTB_volumes_raw_folds <- rsample::group_vfold_cv(
    data = MDTB_volumes_raw_folds_set,
    group = Database,
    balance = "groups",
    v = kfolds
)

# For convenience, keep a copy named 'splits' to inspect
splits <- MDTB_volumes_raw_folds

# Build training sets (analysis data) per fold for downstream steps
MDTB_volumes_raw_folds_list <- purrr::map(
    MDTB_volumes_raw_folds$splits,
    rsample::analysis
) %>% purrr::set_names(paste0("fold_", seq_len(kfolds)))

In [34]:
folds_names <- names(MDTB_volumes_raw_folds_list)

### **1.3.3 Harmonize and prepare data**

Final file:  
**MDTB_volumes_users_folds_harmonized**  
**MDTB_volumes_users_folds_harmonized_long**

In [35]:
# Harmonization volumes controling for Age, Sex
# Note: volumes are already corrected by eTIV

MDTB_volumes_folds_users_harmonized_df <- MDTB_volumes_raw_folds_list %>%
  purrr::imap(~ {
    df  <- .x %>% dplyr::mutate(Database = factor(Database))
    roi <- intersect(MDTB_cerebellar_regions_names, colnames(df))
    mod <- model.matrix(~ Age + Sex + eTIV, data = df)

    combat <- neuroCombat(
      dat        = t(as.matrix(dplyr::select(df, dplyr::all_of(roi)))),
      batch      = df$Database,
      mod        = mod,
      mean.only  = TRUE,
      parametric = TRUE
    )

    vol_harm <- t(combat$dat.combat) %>% tibble::as_tibble()
    colnames(vol_harm) <- roi

    dplyr::bind_cols(dplyr::select(df, -dplyr::all_of(roi)), vol_harm) %>%
      dplyr::select(-tidyselect::any_of(c("eTIV","Sex","Age")))
  }) %>% rlang::set_names(names(MDTB_volumes_raw_folds_list))

[neuroCombat] Found 770 missing data values. 
[neuroCombat] Performing ComBat with mean only
[neuroCombat] Performing ComBat with empirical Bayes
[neuroCombat] Found 54 batches
[neuroCombat] Adjusting for  3  covariate(s) or covariate level(s)
[neuroCombat] Standardizing Data across features
[neuroCombat] Fitting L/S model and finding priors
[neuroCombat] Finding parametric adjustments
[neuroCombat] Adjusting the Data
[neuroCombat] Found 770 missing data values. 
[neuroCombat] Performing ComBat with mean only
[neuroCombat] Performing ComBat with empirical Bayes
[neuroCombat] Found 54 batches
[neuroCombat] Adjusting for  3  covariate(s) or covariate level(s)
[neuroCombat] Standardizing Data across features
[neuroCombat] Fitting L/S model and finding priors
[neuroCombat] Finding parametric adjustments
[neuroCombat] Adjusting the Data
[neuroCombat] Found 770 missing data values. 
[neuroCombat] Performing ComBat with mean only
[neuroCombat] Performing ComBat with empirical Bayes
[neuroComb

In [36]:
MDTB_volumes_users_folds_harmonized <- MDTB_volumes_folds_users_harmonized_df %>% 
    map(~ .x %>%
    mutate(Substance = case_when(Control != 0 ~ Substance, Control == 0 ~ "HC")) %>% 
    filter(!is.na(Substance)) %>% 
    mutate(group = if_else(Substance == "HC",0,1) %>% factor(labels = c("HC","User")),
        group = factor(group, levels = c("User","HC")),
        Substance = factor(Substance, levels = c("ALC","ATS","CANN","COC","NIC","OPI","HC")), .before = 2) %>%
    select(-c(Control)) )

In [37]:
MDTB_volumes_users_folds_harmonized_long <- map(MDTB_volumes_users_folds_harmonized, function(df) {
    pivot_longer(df, names_to = "region", values_to = "volume", MDTB_cerebellar_regions_names) %>%
        arrange(region) %>% mutate(region = factor(region, levels = unique(.$region)))
})

### **1.3.4 Running the analysis for each fold**

### **1.3.3.1 Users**

Final files:  
**p_values_MDTB_folds_users_vs_control**  
**p_values_MDTB_folds_users_vs_control_median**


#### Running the linear models

In [38]:
# Now that we've compared the HC groups lets do it for each HC vs each substance
## Compute the group effect with a linear model (HC as reference)

MDTB_lm_folds_user_vs_control <- map_df(folds_names, function(fold) {
    MDTB_volumes_users_folds_harmonized_long[[fold]] %>%
        select(group, region, volume) %>%
        nest(data = -region) %>%
        mutate(data = map(data, ~ .x %>% mutate(group = relevel(group, ref = "HC")))) %>%
        mutate(test = map(data, ~ {
            m <- lm(volume ~ group, data = .)
            dat <- broom::tidy(m, conf.int = TRUE) %>%
                dplyr::filter(term == "groupUser") %>%
                dplyr::mutate(
                    df = df.residual(m),
                    sigma = sigma(m)
                )
            cont <- emmeans::emmeans(m, specs = "group") %>%
                emmeans::contrast(method = "pairwise")
            cont_df <- broom::tidy(cont)
            cohen_d <- cont_df$estimate / sigma(m)
            dat %>% mutate(effect.size = cohen_d)
        })) %>%
        unnest(test) %>%
        mutate(fold = fold)
})

#### Obtain the null distributions

In [39]:
# calculate the p.values with bootstrapped CI
# Create a list to store all the null distributions

if (!exists("MDTB_nulls_folds_user_vs_control")) {
    n_cols <- length(MDTB_cerebellar_regions_names)
    MDTB_nulls_folds_user_vs_control <- lapply(seq_along(folds_names), function(i) {
        matrix(NA_real_, nrow = num_perm, ncol = n_cols)
    }) %>%
    setNames(folds_names)
}


In [40]:
# Parallel permutation function (foreach) for users vs control

MDTB_compute_null_distributions_folds_users <- function(MDTB_volumes_users_folds_harmonized,
                                                              MDTB_cerebellar_regions_names,
                                                              folds_names,
                                                              MDTB_nulls_folds_user_vs_control,
                                                              num_perm) {
    if (sum(is.na(unlist(MDTB_nulls_folds_user_vs_control))) == 0) return(MDTB_nulls_folds_user_vs_control)
    cl <- parallel::makeCluster(max(1, parallel::detectCores() - 1))
    on.exit(parallel::stopCluster(cl), add = TRUE)
    doParallel::registerDoParallel(cl)

    res <- foreach(i = seq_along(folds_names),
                   .packages = c("dplyr","tidyr","purrr","broom"),
                   .export = c("MDTB_volumes_users_folds_harmonized", "MDTB_cerebellar_regions_names", "folds_names", "MDTB_nulls_folds_user_vs_control", "num_perm")) %dopar% {
        fold_id <- folds_names[i]
        current <- MDTB_nulls_folds_user_vs_control[[fold_id]]
        if (!is.null(current) && sum(is.na(current)) == 0) return(current)

        df <- MDTB_volumes_users_folds_harmonized[[fold_id]] %>%
            dplyr::select(group, dplyr::all_of(MDTB_cerebellar_regions_names))

        n_cols <- length(MDTB_cerebellar_regions_names)
        nulls <- matrix(NA_real_, nrow = num_perm, ncol = n_cols)

        for (j in seq_len(num_perm)) {
            set.seed(42 + i + j)
            boot <- sample.int(nrow(df), replace = TRUE)

            stats <- dplyr::bind_cols(
                    data.frame(group = df$group[boot], check.names = FALSE),
                    dplyr::select(df, dplyr::all_of(MDTB_cerebellar_regions_names))
                ) %>%
                dplyr::mutate(group = stats::relevel(as.factor(group), ref = "HC")) %>%  # set HC as reference
                tidyr::pivot_longer(cols = dplyr::all_of(MDTB_cerebellar_regions_names),
                                    names_to = "region", values_to = "volume") %>%
                tidyr::nest(data = -region) %>%
                dplyr::mutate(test = purrr::map(data, ~ {
                    m <- lm(volume ~ group, data = .x)
                    broom::tidy(m) %>%
                        dplyr::filter(term == "groupUser") %>%  # coefficient for User vs HC
                        dplyr::select(statistic)
                })) %>%
                tidyr::unnest(test) %>%
                dplyr::pull(statistic)

            nulls[j, ] <- stats
        }
        nulls
    } %>% rlang::set_names(folds_names)

    return(res)
}

#### Perform it

In [41]:
dir.create("Statistics/Linear_models/MDTB/RData", showWarnings = FALSE, recursive = TRUE)
# Save/load null distributions for users vs control for each fold

nulls_fp_folds_users <- file.path("Statistics", "Linear_models", "MDTB", "RData",
                 "MDTB_nulls_folds_user_vs_control.RData")

if (!file.exists(nulls_fp_folds_users)) {
    MDTB_nulls_folds_user_vs_control <- MDTB_compute_null_distributions_folds_users(
        MDTB_volumes_users_folds_harmonized,
        MDTB_cerebellar_regions_names,
        folds_names,
        MDTB_nulls_folds_user_vs_control,
        num_perm
        )

    # Ensure null matrices have region column names
    MDTB_nulls_folds_user_vs_control <- purrr::imap(
      MDTB_nulls_folds_user_vs_control,
      ~ { colnames(.x) <- MDTB_cerebellar_regions_names; .x }
    )

    # Build CIs per fold x region from null distributions
    MDTB_folds_CIs <- purrr::imap_dfr(
      MDTB_nulls_folds_user_vs_control,
      ~ {
        low  <- apply(.x, 2, quantile, probs = 0.025)
        high <- apply(.x, 2, quantile, probs = 0.975)
        tibble::tibble(
          fold          = .y,
          region        = names(low),
          perm_CI_lower = as.numeric(low),
          perm_CI_upper = as.numeric(high)
        )
      }
    )

    # Compute permutation p-values per fold and merge with CIs
    p_values_MDTB_folds_users_vs_control <- MDTB_lm_folds_user_vs_control %>%
      dplyr::select(-data) %>%
      dplyr::mutate(
        p_value = purrr::pmap_dbl(
          list(fold, as.character(region), statistic),
          function(fold_id, roi, stat) {
            null_distribution <- MDTB_nulls_folds_user_vs_control[[fold_id]][, roi]
            mean(abs(null_distribution) >= abs(stat))
          }
        )
      ) %>%
      dplyr::group_by(fold) %>%
      dplyr::mutate(p_adj = p.adjust(p_value, method = "fdr")) %>%
      dplyr::ungroup() %>%
      dplyr::left_join(MDTB_folds_CIs, by = c("fold", "region"))

save(MDTB_nulls_folds_user_vs_control,
        p_values_MDTB_folds_users_vs_control,
        file = nulls_fp_folds_users)
} else {
load(nulls_fp_folds_users)
}


#### Obtain the mean, median, mode for all folds

In [42]:
Mode <- function(x) {
    ux <- unique(x)
    ux[which.max(tabulate(match(x, ux)))]
}

In [43]:
p_values_MDTB_folds_users_vs_control_median <- p_values_MDTB_folds_users_vs_control %>%
    group_by(region) %>%
    summarise(effect.size_median = median(effect.size,na.rm = TRUE),
        statistic_median = median(statistic, na.rm = TRUE),
        p_value_median = median(p_value, na.rm = TRUE),
        p_adj_median = median(p_adj, na.rm = TRUE),
        statistic_mode = Mode(statistic),
        effect.size_mode = Mode(effect.size),
        p_value_mode = Mode(p_value),
        p_adj_mode = Mode(p_adj),
        statistic_media = mean(statistic, na.rm = TRUE),
        effect.size_media = mean(effect.size, na.rm = TRUE),
        p_value_media = mean(p_value, na.rm = TRUE),
        p_adj_media = mean(p_adj, na.rm = TRUE),
        perm_CI_lower = mean(perm_CI_lower, na.rm = TRUE),
        perm_CI_upper = mean(perm_CI_upper, na.rm = TRUE),
        estimate_median = median(estimate, na.rm = TRUE),
        estimate_mode = Mode(estimate),
        estimate_media = mean(estimate, na.rm = TRUE)
        ) %>%
    ungroup()


### **1.3.3.2 Substances**

Final files:  
**p_values_MDTB_folds_substance_vs_control**  
**p_values_MDTB_folds_substance_vs_control_median**

#### Running the linear models

In [44]:
dir.create("Statistics/Linear_models/MDTB/RData", showWarnings = FALSE, recursive = TRUE)

lm_fp_folds_substances <- file.path(
    "Statistics", "Linear_models", "MDTB", "RData",
    "MDTB_lm_folds_substance_vs_control.RData"
)

if (!file.exists(lm_fp_folds_substances)) {
    # Now that we've compared the HC groups lets do it for each HC vs each substance
    # Compute the group effect with a linear model (HC as reference)
    MDTB_lm_folds_substance_vs_control <- purrr::map_df(substances_names, function(substance) {
        df_substance <- MDTB_volumes_users_folds_harmonized_long %>%
            purrr::map(~ dplyr::filter(.x, Substance == "HC" | Substance == substance))

        purrr::map_df(names(df_substance), function(fold) {
            df_fold <- df_substance[[fold]]
            df_fold %>%
                dplyr::select(group, region, volume) %>%
                tidyr::nest(data = -region) %>%
                dplyr::mutate(data = purrr::map(data, ~ dplyr::mutate(.x, group = relevel(group, ref = "HC")))) %>%
                dplyr::mutate(test = purrr::map(data, ~ {
                    m <- lm(volume ~ group, data = .)
                    dat <- broom::tidy(m, conf.int = TRUE) %>%
                        dplyr::filter(term == "groupUser") %>%
                        dplyr::mutate(
                            df = df.residual(m),
                            sigma = sigma(m)
                        )
                    cont <- emmeans::emmeans(m, specs = "group") %>%
                        emmeans::contrast(method = "pairwise")
                    cont_df <- broom::tidy(cont)
                    cohen_d <- cont_df$estimate / sigma(m)
                    dat %>% dplyr::mutate(effect.size = cohen_d)
                })) %>%
                tidyr::unnest(test) %>%
                dplyr::mutate(
                    Substance = substance,
                    fold = fold
                )
        })
    })

    save(MDTB_lm_folds_substance_vs_control, file = lm_fp_folds_substances)
} else {
    load(lm_fp_folds_substances)
}

#### Obtain the null distributions

In [45]:
# calculate the p.values with bootstrapped CI
# Create a list to store all the null distributions

if (!exists("MDTB_nulls_folds_substance_vs_control")){
    n_cols <- length(MDTB_cerebellar_regions_names)
    MDTB_nulls_folds_substance_vs_control <- lapply(substances_names, function(substance) {
        lapply(seq_along(folds_names), function(i) {
            matrix(NA_real_, nrow = num_perm, ncol = n_cols)
        }) %>%
        setNames(folds_names)
    }) %>%
    setNames(substances_names)
}

In [46]:
# Parallel permutation function (foreach) for each substance vs control across folds

MDTB_compute_null_distributions_folds_substances <- function(
  MDTB_volumes_users_folds_harmonized,
  MDTB_cerebellar_regions_names,
  substances_names,
  folds_names,
  MDTB_nulls_folds_substance_vs_control,
  num_perm
) {
  if (sum(is.na(unlist(MDTB_nulls_folds_substance_vs_control))) == 0) {
  return(MDTB_nulls_folds_substance_vs_control)
  }

  cl <- parallel::makeCluster(max(1, parallel::detectCores() - 1))
  on.exit(parallel::stopCluster(cl), add = TRUE)
  doParallel::registerDoParallel(cl)

  res <- foreach(i = seq_along(substances_names),
         .packages = c("dplyr","tidyr","purrr","broom"),
         .export = c("MDTB_volumes_users_folds_harmonized", "MDTB_cerebellar_regions_names", "substances_names", "folds_names", "MDTB_nulls_folds_substance_vs_control", "num_perm")) %dopar% {
  substance <- substances_names[i]
  current_sub <- MDTB_nulls_folds_substance_vs_control[[substance]]

  fold_results <- vector("list", length(folds_names))
  names(fold_results) <- folds_names

  for (f in seq_along(folds_names)) {
    fold_id <- folds_names[f]
    current <- if (!is.null(current_sub)) current_sub[[fold_id]] else NULL
    if (!is.null(current) && sum(is.na(current)) == 0) {
    fold_results[[fold_id]] <- current
    next
    }

    df <- MDTB_volumes_users_folds_harmonized[[fold_id]] %>%
    dplyr::filter(Substance %in% c("HC", substance)) %>%
    dplyr::select(group, dplyr::all_of(MDTB_cerebellar_regions_names))

    n_cols <- length(MDTB_cerebellar_regions_names)
    nulls <- matrix(NA_real_, nrow = num_perm, ncol = n_cols)

    for (j in seq_len(num_perm)) {
    set.seed(42 + i + f + j)
    boot <- sample.int(nrow(df), replace = TRUE)

    stats <- dplyr::bind_cols(
      data.frame(group = df$group[boot], check.names = FALSE),
      dplyr::select(df, dplyr::all_of(MDTB_cerebellar_regions_names))
      ) %>%
      dplyr::mutate(group = stats::relevel(as.factor(group), ref = "HC")) %>%  # set HC as reference
      tidyr::pivot_longer(cols = dplyr::all_of(MDTB_cerebellar_regions_names),
                names_to = "region", values_to = "volume") %>%
      tidyr::nest(data = -region) %>%
      dplyr::mutate(test = purrr::map(data, ~ {
      m <- lm(volume ~ group, data = .x)
      broom::tidy(m) %>%
        dplyr::filter(term == "groupUser") %>%  # coefficient for User vs HC
        dplyr::select(statistic)
      })) %>%
      tidyr::unnest(test) %>%
      dplyr::pull(statistic)

    nulls[j, ] <- stats
    }
    fold_results[[fold_id]] <- nulls
  }
  fold_results
  } %>% rlang::set_names(substances_names)

  return(res)
}

#### Perform it

In [47]:

dir.create("Statistics/Linear_models/MDTB/RData", showWarnings = FALSE, recursive = TRUE)
nulls_fp_folds_substances <- file.path("Statistics","Linear_models","MDTB","RData",
                                       "MDTB_nulls_folds_substance_vs_control.RData")

if (!file.exists(nulls_fp_folds_substances)) {
    MDTB_nulls_folds_substance_vs_control <- MDTB_compute_null_distributions_folds_substances(
        MDTB_volumes_users_folds_harmonized,
        MDTB_cerebellar_regions_names,
        substances_names,
        folds_names,
        MDTB_nulls_folds_substance_vs_control,
        num_perm
    )

    # Build CIs per fold x region from null distributions
    MDTB_folds_CIs_substances <- purrr::imap_dfr(
      MDTB_nulls_folds_substance_vs_control,
      ~ {
        substance <- .y
        fold_list <- .x
        purrr::imap_dfr(fold_list, function(mat, fold_id) {
          low  <- apply(mat, 2, quantile, probs = 0.025)
          high <- apply(mat, 2, quantile, probs = 0.975)
          tibble::tibble(
            Substance     = substance,
            fold          = fold_id,
            region        = names(low),
            perm_CI_lower = as.numeric(low),
            perm_CI_upper = as.numeric(high)
          )
        })
      }
    )

    # Compute permutation p-values per fold and merge with CIs
    # Ensure null matrices have region column names
    MDTB_nulls_folds_substance_vs_control <- purrr::imap(
      MDTB_nulls_folds_substance_vs_control,
      ~ purrr::imap(.x, function(mat, fold_id) {
      colnames(mat) <- MDTB_cerebellar_regions_names
      mat
      })
    )

    p_values_MDTB_folds_substance_vs_control <- MDTB_lm_folds_substance_vs_control %>%
      dplyr::select(-data) %>%
      dplyr::mutate(
      p_value = purrr::pmap_dbl(
        list(Substance, fold, as.character(region), statistic),
        function(substance, fold_id, roi, stat) {
        null_distribution <- MDTB_nulls_folds_substance_vs_control[[substance]][[fold_id]][, roi]
        mean(abs(null_distribution) >= abs(stat))
        }
      )) %>%
      dplyr::group_by(Substance, fold) %>%
      dplyr::mutate(p_adj = p.adjust(p_value, method = "fdr")) %>%
      dplyr::ungroup() %>%
      dplyr::left_join(MDTB_folds_CIs_substances, by = c("Substance", "fold"))

    save(MDTB_nulls_folds_substance_vs_control,
       MDTB_folds_CIs_substances,
       p_values_MDTB_folds_substance_vs_control,
       file = nulls_fp_folds_substances)

} else {
    load(nulls_fp_folds_substances)
}


#### Obtain the mean, median and mode for all folds

In [48]:
p_values_MDTB_folds_substance_vs_control_median <- p_values_MDTB_folds_substance_vs_control %>%
    group_by(Substance, region) %>%
    summarise(effect.size_median = median(effect.size,na.rm = TRUE),
        statistic_median = median(statistic, na.rm = TRUE),
        p_value_median = median(p_value, na.rm = TRUE),
        p_adj_median = median(p_adj, na.rm = TRUE),
        statistic_mode = Mode(statistic),
        effect.size_mode = Mode(effect.size),
        p_value_mode = Mode(p_value),
        p_adj_mode = Mode(p_adj),
        statistic_media = mean(statistic, na.rm = TRUE),
        effect.size_media = mean(effect.size, na.rm = TRUE),
        p_value_media = mean(p_value, na.rm = TRUE),
        p_adj_media = mean(p_adj, na.rm = TRUE),
        perm_CI_lower = mean(perm_CI_lower, na.rm = TRUE),
        perm_CI_upper = mean(perm_CI_upper, na.rm = TRUE),
        estimate_median = median(estimate, na.rm = TRUE),
        estimate_mode = Mode(estimate),
        estimate_media = mean(estimate, na.rm = TRUE)
        ) %>%
    ungroup()



`summarise()` has grouped output by 'Substance'. You can override using the
`.groups` argument.


## **Save**

In [49]:
stats_fp <- "Statistics/Linear_models/MDTB/MDTB_volumes_statistics.RData"

if (!file.exists(stats_fp)) {
    save(MDTB_volumes,
             MDTB_volumes_long,
             p_values_MDTB_user_vs_control,
             p_values_MDTB_substance_vs_control,
             MDTB_volumes_matchit_users_df,
             MDTB_volumes_matchit_users_harmonized_df,
             MDTB_volumes_matchit_users_harmonized_long,
             p_values_MDTB_matchit_users_vs_control,
             MDTB_volumes_matchit_substances_df,
             MDTB_volumes_matchit_substances_harmonized_df,
             MDTB_volumes_matchit_substances_harmonized_long,
             p_values_MDTB_matchit_substance_vs_control,
             MDTB_volumes_raw_folds_list,
             MDTB_volumes_users_folds_harmonized,
             MDTB_volumes_users_folds_harmonized_long,
             p_values_MDTB_folds_users_vs_control,
             p_values_MDTB_folds_users_vs_control_median,
             p_values_MDTB_folds_substance_vs_control,
             p_values_MDTB_folds_substance_vs_control_median,
             plot_subs_per_DB,
             file = stats_fp)
} else {
    message("File exists, skipping: ", stats_fp)
}

File exists, skipping: Statistics/Linear_models/MDTB/MDTB_volumes_statistics.RData



---
## **Sanity Check: Sensitivity analyses on complete-cases (CC), alternative-imputed (AI) datasets and Covariate-Adjusted (CA)**

The primary analyses above used Age and Sex imputed with `eTIV` as a predictor (potentially circular).  
These sensitivity analyses re-run the **User vs HC** and **Substance vs HC** linear models on:

- **CC** — subjects with fully observed Age and Sex (no imputation)
- **AI** — Age and Sex imputed using Substance group and Site only (no neuroanatomical predictors)
- **CA** - models fit with explicit covariate adjustment (**Age + Sex + eTIV**) in the linear model (instead of relying on imputation/harmonization choices)

*Note: Standard FDR-corrected p-values are used here (not permutation), which is appropriate for a sensitivity comparison.*


In [50]:

# Prepare CC and AI MDTB volumes for sensitivity analyses
if (!exists("Volumes_Cerebellum_MDTB_Harmonized_CC")) {
    load("Data/RData/Sanity_check/Volumes_Cerebellum_Sanity_check.RData")
}

# Load CA (Covariate-Adjusted) data
load("Data/RData/Volumes_Cerebellum_Covariate_Adjusted.RData")
load("Data/RData/Volumes_Cerebellum_MDTB_Covariate_Adjusted.RData")

prepare_mdtb_sc <- function(harmonized_df) {
    harmonized_df %>%
        filter(Substance != "OPI") %>%
        mutate(
            group     = factor(if_else(Substance == "HC", 0, 1), labels = c("HC", "User")),
            Substance = factor(Substance, levels = c("ALC","ATS","CANN","COC","NIC","HC","OPI"))
        ) %>%
        mutate(group = factor(group, levels = c("User","HC"))) %>%
        mutate(Database = factor(Database))
}
MDTB_volumes_CC <- prepare_mdtb_sc(Volumes_Cerebellum_MDTB_Harmonized_CC)
MDTB_volumes_AI <- prepare_mdtb_sc(Volumes_Cerebellum_MDTB_Harmonized_AI)
MDTB_volumes_CA <- prepare_mdtb_sc(Volumes_Cerebellum_MDTB_Harmonized_CA)

MDTB_cerebellar_regions_names_sc <- MDTB_volumes_CC %>%
    select(contains("region")) %>% colnames()

MDTB_volumes_long_CC <- MDTB_volumes_CC %>%
    pivot_longer(names_to = "region", values_to = "volume",
                 all_of(MDTB_cerebellar_regions_names_sc))

MDTB_volumes_long_AI <- MDTB_volumes_AI %>%
    pivot_longer(names_to = "region", values_to = "volume",
                 all_of(MDTB_cerebellar_regions_names_sc))
MDTB_volumes_long_CA <- MDTB_volumes_CA %>%
    pivot_longer(names_to = "region", values_to = "volume",
                 all_of(MDTB_cerebellar_regions_names_sc))



cat("MDTB CC dataset:", nrow(MDTB_volumes_CC), "subjects\n")

cat("MDTB CC dataset:", nrow(MDTB_volumes_CC), "subjects\n")
cat("MDTB AI dataset:", nrow(MDTB_volumes_AI), "subjects\n")
cat("MDTB CA dataset:", nrow(MDTB_volumes_CA), "subjects\n")

cat("MDTB AI dataset:", nrow(MDTB_volumes_AI), "subjects\n")

MDTB CC dataset: 2933 subjects
MDTB CC dataset: 2933 subjects
MDTB AI dataset: 3461 subjects
MDTB CA dataset: 2582 subjects
MDTB AI dataset: 3461 subjects


In [51]:

# ---- CC: User vs HC (MDTB) ----
p_values_MDTB_CC_user_vs_control <- MDTB_volumes_long_CC %>%
    select(group, region, volume) %>%
    nest(data = -region) %>%
    mutate(data = map(data, ~ .x %>% mutate(group = relevel(group, ref = "HC")))) %>%
    mutate(test = map(data, ~ {
        m <- lm(volume ~ group, data = .)
        broom::tidy(m, conf.int = TRUE) %>%
            filter(term == "groupUser") %>%
            mutate(sigma = sigma(m), effect.size = estimate / sigma(m),
                   df = df.residual(m))
    })) %>%
    unnest(test) %>%
    select(-data) %>%
    mutate(p.adj = p.adjust(p.value, method = "fdr"))


In [52]:

# ---- CC: substance vs HC (MDTB) ----
substances_names_CC_mdtb <- unique(MDTB_volumes_CC$Substance)[
    unique(MDTB_volumes_CC$Substance) != "HC"] %>% as.vector()

p_values_MDTB_CC_substance_vs_control <- map_df(substances_names_CC_mdtb, .f = ~ {
    MDTB_volumes_long_CC %>%
        filter(Substance %in% c("HC", .x)) %>%
        select(group, region, volume) %>%
        nest(data = -region) %>%
        mutate(data = map(data, ~ .x %>% mutate(group = relevel(group, ref = "HC")))) %>%
        mutate(test = map(data, ~ {
            m <- lm(volume ~ group, data = .)
            broom::tidy(m, conf.int = TRUE) %>%
                filter(term == "groupUser") %>%
                mutate(sigma = sigma(m), effect.size = estimate / sigma(m),
                       df = df.residual(m))
        })) %>%
        unnest(test) %>%
        select(-data) %>%
        mutate(Substance = .x)
}) %>%
    mutate(p.adj = p.adjust(p.value, method = "fdr"))


In [53]:

# ---- AI: User vs HC (MDTB) ----
p_values_MDTB_AI_user_vs_control <- MDTB_volumes_long_AI %>%
    select(group, region, volume) %>%
    nest(data = -region) %>%
    mutate(data = map(data, ~ .x %>% mutate(group = relevel(group, ref = "HC")))) %>%
    mutate(test = map(data, ~ {
        m <- lm(volume ~ group, data = .)
        broom::tidy(m, conf.int = TRUE) %>%
            filter(term == "groupUser") %>%
            mutate(sigma = sigma(m), effect.size = estimate / sigma(m),
                   df = df.residual(m))
    })) %>%
    unnest(test) %>%
    select(-data) %>%
    mutate(p.adj = p.adjust(p.value, method = "fdr"))


In [54]:

# ---- AI: substance vs HC (MDTB) ----
substances_names_AI_mdtb <- unique(MDTB_volumes_AI$Substance)[
    unique(MDTB_volumes_AI$Substance) != "HC"] %>% as.vector()

p_values_MDTB_AI_substance_vs_control <- map_df(substances_names_AI_mdtb, .f = ~ {
    MDTB_volumes_long_AI %>%
        filter(Substance %in% c("HC", .x)) %>%
        select(group, region, volume) %>%
        nest(data = -region) %>%
        mutate(data = map(data, ~ .x %>% mutate(group = relevel(group, ref = "HC")))) %>%
        mutate(test = map(data, ~ {
            m <- lm(volume ~ group, data = .)
            broom::tidy(m, conf.int = TRUE) %>%
                filter(term == "groupUser") %>%
                mutate(sigma = sigma(m), effect.size = estimate / sigma(m),
                       df = df.residual(m))
        })) %>%
        unnest(test) %>%
        select(-data) %>%
        mutate(Substance = .x)
}) %>%
    mutate(p.adj = p.adjust(p.value, method = "fdr"))


In [55]:
# ---- CA: User vs HC (MDTB) ----
p_values_MDTB_CA_user_vs_control <- MDTB_volumes_long_CA %>%
    select(group, region, volume) %>%
    nest(data = -region) %>%
    mutate(data = map(data, ~ .x %>% mutate(group = relevel(group, ref = "HC")))) %>%
    mutate(test = map(data, ~ {
        m <- lm(volume ~ group, data = .)
        broom::tidy(m, conf.int = TRUE) %>%
            filter(term == "groupUser") %>%
            mutate(sigma = sigma(m), effect.size = estimate / sigma(m),
                   df = df.residual(m))
    })) %>%
    unnest(test) %>%
    select(-data) %>%
    mutate(p.adj = p.adjust(p.value, method = "fdr"))

# ---- CA: substance vs HC (MDTB) ----
substances_names_CA_mdtb <- unique(MDTB_volumes_CA$Substance)[
    unique(MDTB_volumes_CA$Substance) != "HC"] %>% as.vector()

p_values_MDTB_CA_substance_vs_control <- map_df(substances_names_CA_mdtb, .f = ~ {
    MDTB_volumes_long_CA %>%
        filter(Substance %in% c("HC", .x)) %>%
        select(group, region, volume) %>%
        nest(data = -region) %>%
        mutate(data = map(data, ~ .x %>% mutate(group = relevel(group, ref = "HC")))) %>%
        mutate(test = map(data, ~ {
            m <- lm(volume ~ group, data = .)
            broom::tidy(m, conf.int = TRUE) %>%
                filter(term == "groupUser") %>%
                mutate(sigma = sigma(m), effect.size = estimate / sigma(m),
                       df = df.residual(m))
        })) %>%
        unnest(test) %>%
        select(-data) %>%
        mutate(Substance = .x)
}) %>%
    mutate(p.adj = p.adjust(p.value, method = "fdr"))

In [56]:

# Save MDTB sanity check results
dir.create("Statistics/Sanity_check/MDTB", recursive = TRUE, showWarnings = FALSE)
sc_fp_mdtb <- "Statistics/Sanity_check/MDTB/MDTB_sanity_check_statistics.RData"

if (!file.exists(sc_fp_mdtb)) {
    save(p_values_MDTB_CC_user_vs_control,
         p_values_MDTB_CC_substance_vs_control,
         p_values_MDTB_AI_user_vs_control,
         p_values_MDTB_AI_substance_vs_control,
         p_values_MDTB_CA_user_vs_control,
         p_values_MDTB_CA_substance_vs_control,
         file = sc_fp_mdtb)
} else {
    message("File exists, skipping: ", sc_fp_mdtb)
}

purrr::iwalk(
    list(
        MDTB_CC_user_vs_control       = p_values_MDTB_CC_user_vs_control,
        MDTB_CC_substance_vs_control  = p_values_MDTB_CC_substance_vs_control,
        MDTB_AI_user_vs_control       = p_values_MDTB_AI_user_vs_control,
        MDTB_AI_substance_vs_control  = p_values_MDTB_AI_substance_vs_control,
        MDTB_CA_user_vs_control       = p_values_MDTB_CA_user_vs_control,
        MDTB_CA_substance_vs_control  = p_values_MDTB_CA_substance_vs_control
    ),
    ~ write_csv(.x, file.path("Statistics/Sanity_check/MDTB", paste0(.y, ".csv")))
)
